In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# -----------------------------
# 1. Hyperparameters
# -----------------------------
batch_size = 64
learning_rate = 0.001
epochs = 10
hidden_layers = [256, 128]  # Tunable
activation_fn = nn.ReLU()   # Try nn.Sigmoid(), nn.Tanh(), etc.
dataset_choice = ""    # Change to "FashionMNIST"

# -----------------------------
# 2. Load Dataset
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

if dataset_choice == "MNIST":
    train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
else:
    train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# -----------------------------
# 3. Define MLP Model
# -----------------------------
class MLP(nn.Module):
    def __init__(self, input_size, hidden_layers, num_classes, activation):
        super(MLP, self).__init__()
        layers = []
        in_features = input_size
        for h in hidden_layers:
            layers.append(nn.Linear(in_features, h))
            layers.append(activation)
            in_features = h
        layers.append(nn.Linear(in_features, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        return self.network(x)

model = MLP(28*28, hidden_layers, 10, activation_fn)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# -----------------------------
# 4. Loss & Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# -----------------------------
# 5. Training Loop
# -----------------------------
train_losses, test_losses, accuracies = [], [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_losses.append(running_loss / len(train_loader))

    # -----------------------------
    # 6. Evaluate on Test Set
    # -----------------------------
    model.eval()
    correct, total, test_loss = 0, 0, 0.0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    accuracies.append(accuracy)
    test_losses.append(test_loss / len(test_loader))

    print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_losses[-1]:.4f}, Test Loss: {test_losses[-1]:.4f}, Accuracy: {accuracy:.2f}%")

# -----------------------------
# 7. Plot Loss & Accuracy Curves
# -----------------------------
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1,2,2)
plt.plot(accuracies, label='Test Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Curve')
plt.legend()
plt.show()

100%|██████████| 26.4M/26.4M [00:01<00:00, 20.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 333kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 6.22MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 21.1MB/s]


Epoch [1/10], Train Loss: 0.4961, Test Loss: 0.4236, Accuracy: 84.95%
Epoch [2/10], Train Loss: 0.3685, Test Loss: 0.4199, Accuracy: 84.12%
Epoch [3/10], Train Loss: 0.3280, Test Loss: 0.3623, Accuracy: 86.77%
Epoch [4/10], Train Loss: 0.3034, Test Loss: 0.3583, Accuracy: 86.58%


In [ ]:
'''
1. Import Libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader


torch: Core PyTorch library.

nn: Neural network layers (Linear, ReLU, etc.).

optim: Optimization algorithms (Adam, SGD, etc.).

torchvision: Datasets & image transformations.

transforms: Preprocessing pipelines for images.

matplotlib.pyplot: For plotting loss & accuracy.

DataLoader: Creates data batches for training/testing.

2. Hyperparameters
batch_size = 64
learning_rate = 0.001
epochs = 10
hidden_layers = [256, 128]  # Tunable
activation_fn = nn.ReLU()   # Try nn.Sigmoid(), nn.Tanh(), etc.
dataset_choice = ""    # Change to "FashionMNIST"


batch_size: How many images per training batch.

learning_rate: Step size for gradient descent.

epochs: Full passes over training set.

hidden_layers: Architecture of the MLP (2 hidden layers).

activation_fn: Non-linearity applied after each layer.

dataset_choice: Can switch between MNIST and FashionMNIST.

Concept:
A multi-layer perceptron (MLP) is a sequence of Linear layers + activation functions.

3. Load Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


ToTensor() converts images from PIL to PyTorch tensors.

Normalize(mean, std): scales pixel values.

Dataset Selection
if dataset_choice == "MNIST":
    train_dataset = ...
else:
    train_dataset = torchvision.datasets.FashionMNIST(...)


Downloads MNIST or FashionMNIST.

train=True loads training split, train=False loads test split.

Create Data Loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


Loads data in batches.

shuffle=True randomizes training data for better learning.

4. Define the MLP Model
class MLP(nn.Module):
    def __init__(self, input_size, hidden_layers, num_classes, activation):
        super(MLP, self).__init__()
        layers = []
        in_features = input_size


Creating a class MLP that inherits from nn.Module.

input_size = 28*28 because images are 28×28 pixels (flattened).

layers = [] a list that will hold all layers.

Create Hidden Layers Dynamically
for h in hidden_layers:
    layers.append(nn.Linear(in_features, h))
    layers.append(activation)
    in_features = h


This loop builds:

First hidden layer: 784 → 256

Second hidden layer: 256 → 128

Activation function after each layer.

Add Output Layer
layers.append(nn.Linear(in_features, num_classes))
self.network = nn.Sequential(*layers)


Final output 128 → 10 classes.

Concept:
nn.Sequential stacks layers so they execute in order.

Define Forward Pass
def forward(self, x):
    x = x.view(x.size(0), -1)  # Flatten
    return self.network(x)


Converts (batch, 1, 28, 28) → (batch, 784)

Sends through neural network.

Create Model and Set Device
model = MLP(28*28, hidden_layers, 10, activation_fn)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


Creates model instance.

Moves model to GPU if available.

5. Loss & Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


CrossEntropyLoss = standard classification loss.

Adam = adaptive gradient optimizer.

6. Training Loop
train_losses, test_losses, accuracies = [], [], []


Stores values for plotting later.

Epoch Loop
for epoch in range(epochs):
    model.train()
    running_loss = 0.0


model.train() enables dropout, gradients, etc.

running_loss accumulates batch losses.

Batch Loop
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)


Load batch.

Move batch to GPU/CPU.

Forward Pass
outputs = model(images)
loss = criterion(outputs, labels)


Compute predictions.

Compute loss.

Backward Pass
optimizer.zero_grad()
loss.backward()
optimizer.step()


Clear previous gradients.

Compute new gradients.

Update parameters using Adam.

Store Train Loss
running_loss += loss.item()


Convert tensor → Python float.

Save Average Loss
train_losses.append(running_loss / len(train_loader))

7. Evaluate on Test Set
model.eval()
correct, total, test_loss = 0, 0, 0.0


model.eval() disables dropout, batchnorm randomness.

Disable Gradients
with torch.no_grad():


Saves memory and computation.

Test Loop
outputs = model(images)
loss = criterion(outputs, labels)


Compute test loss.

Predictions
_, predicted = torch.max(outputs, 1)
total += labels.size(0)
correct += (predicted == labels).sum().item()


Concept:
torch.max(outputs,1) returns the index of the highest logit → predicted class.

Compute Accuracy
accuracy = 100 * correct / total
accuracies.append(accuracy)
test_losses.append(test_loss / len(test_loader))

Print Epoch Results
print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {train_losses[-1]:.4f}, Test Loss: {test_losses[-1]:.4f}, Accuracy: {accuracy:.2f}%")

8. Plot Loss & Accuracy
plt.figure(figsize=(12,5))


Creates two subplots.

Plot Loss Curves
plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')

Plot Accuracy Curve
plt.subplot(1,2,2)
plt.plot(accuracies, label='Test Accuracy')

plt.show()


Display the plots.

🎉 Final Summary (Easy to Remember)
Part	What it does
Dataset	Loads MNIST or FashionMNIST
Model	Builds MLP with dynamic hidden layers
Training	Forward → Loss → Backprop → Update
Evaluation	Compute accuracy & test loss
Visualization	Plots loss + accuracy curves

SyntaxError: incomplete input (ipython-input-3083282452.py, line 1)